# Execucao da camada gold de indicadores financeiros

Este notebook reconstrói a cadeia principal da camada `gold` na ordem correta:

1. remove as views dependentes da mart;
2. recria `layer_03_gold.mart_indicadores_financeiros`;
3. recria as views de benchmark e IPRF.

Isso evita o erro `DependentObjectsStillExist` ao tentar derrubar a mart enquanto ainda existem views que dependem dela.


## 1. Setup


In [1]:
from __future__ import annotations

import os
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'queries').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do repositorio.')


ROOT = find_repo_root()
ENV_PATH = ROOT / '.env'
QUERIES_DIR = ROOT / 'queries' / '03_gold'

SQL_FILES = [
    QUERIES_DIR / '01_refaz_mart_indicadores_financeiros_todas_empresas.sql',
    QUERIES_DIR / '02_create_vw_benchmark_indicadores_anual.sql',
    QUERIES_DIR / '03_create_vw_iprf_empresas_anual.sql',
    QUERIES_DIR / '04_create_vw_benchmark_iprf_setorial.sql',
]

load_dotenv(ENV_PATH)

raw_db_user = os.getenv('DB_USER', 'postgres')
raw_db_pass = os.getenv('DB_PASS') or os.getenv('DB_PASSWORD') or 'password'
DB_USER = quote_plus(raw_db_user)
DB_PASS = quote_plus(raw_db_pass)
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'data_lake')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

print(f'Raiz do projeto: {ROOT}')
print(f'Banco alvo: {DB_HOST}:{DB_PORT}/{DB_NAME}')
print('Arquivos SQL que serao executados:')
for sql_path in SQL_FILES:
    print(f' - {sql_path}')


Raiz do projeto: c:\Users\renan\Documents\FAE\Big Data for Finance\big_data_for_finance
Banco alvo: localhost:5433/bigdata_for_finance
Arquivos SQL que serao executados:
 - c:\Users\renan\Documents\FAE\Big Data for Finance\big_data_for_finance\queries\03_gold\01_refaz_mart_indicadores_financeiros_todas_empresas.sql
 - c:\Users\renan\Documents\FAE\Big Data for Finance\big_data_for_finance\queries\03_gold\02_create_vw_benchmark_indicadores_anual.sql
 - c:\Users\renan\Documents\FAE\Big Data for Finance\big_data_for_finance\queries\03_gold\03_create_vw_iprf_empresas_anual.sql
 - c:\Users\renan\Documents\FAE\Big Data for Finance\big_data_for_finance\queries\03_gold\04_create_vw_benchmark_iprf_setorial.sql


## 2. Derrubar dependencias e recriar os objetos

Primeiro removemos as views na ordem inversa da dependencia. Depois executamos as SQLs da mart e das views novamente.


In [2]:
PRE_DROP_SQL = '''
DROP VIEW IF EXISTS "layer_03_gold"."vw_benchmark_iprf_setorial";
DROP VIEW IF EXISTS "layer_03_gold"."vw_iprf_empresas_anual";
DROP VIEW IF EXISTS "layer_03_gold"."vw_benchmark_indicadores_anual";
'''


def execute_sql_batch(raw_conn, sql_text: str, label: str) -> None:
    with raw_conn.cursor() as cur:
        cur.execute(sql_text)
    raw_conn.commit()
    print(f'OK - {label}')


raw_conn = engine.raw_connection()
try:
    execute_sql_batch(raw_conn, PRE_DROP_SQL, 'views dependentes removidas')

    for sql_path in SQL_FILES:
        sql_text = sql_path.read_text(encoding='utf-8')
        execute_sql_batch(raw_conn, sql_text, f'executado {sql_path.name}')

    print('OK - cadeia gold reconstruida com sucesso')
except Exception:
    raw_conn.rollback()
    raise
finally:
    raw_conn.close()


OK - views dependentes removidas
OK - executado 01_refaz_mart_indicadores_financeiros_todas_empresas.sql
OK - executado 02_create_vw_benchmark_indicadores_anual.sql
OK - executado 03_create_vw_iprf_empresas_anual.sql
OK - executado 04_create_vw_benchmark_iprf_setorial.sql
OK - cadeia gold reconstruida com sucesso


## 3. Confirmacao rapida


In [3]:
q_count = '''
SELECT COUNT(*) AS n_linhas
FROM "layer_03_gold"."mart_indicadores_financeiros";
'''

q_views = '''
SELECT table_name
FROM information_schema.views
WHERE table_schema = 'layer_03_gold'
  AND table_name IN (
    'vw_benchmark_indicadores_anual',
    'vw_iprf_empresas_anual',
    'vw_benchmark_iprf_setorial'
  )
ORDER BY table_name;
'''

q_sample = '''
SELECT "CNPJ_CIA", "DENOM_CIA", "ANO_FISCAL", "SCORE_IPRF", "FAIXA_IPRF"
FROM "layer_03_gold"."vw_iprf_empresas_anual"
ORDER BY "ANO_FISCAL" DESC, "DENOM_CIA"
LIMIT 10;
'''

df_count = pd.read_sql(text(q_count), engine)
df_views = pd.read_sql(text(q_views), engine)
df_sample = pd.read_sql(text(q_sample), engine)

display(df_count)
display(df_views)
display(df_sample)


,n_linhas
0,2218


,table_name
0,vw_benchmark_indicadores_anual
1,vw_benchmark_iprf_setorial
2,vw_iprf_empresas_anual


,CNPJ_CIA,DENOM_CIA,ANO_FISCAL,SCORE_IPRF,FAIXA_IPRF
0,07.628.528/0001-59,BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,2025,3.315870,Alerta
1,64.904.295/0001-03,CAMIL ALIMENTOS S.A.,2025,4.413642,Alerta
2,06.981.381/0001-13,CTC - CENTRO DE TECNOLOGIA CANAVIEIRA S.A.,2025,8.560458,Saudavel
3,02.635.522/0001-95,JALLES MACHADO S.A.,2025,5.177543,Moderado
4,88.613.658/0001-10,PETTENATI S.A. INDUSTRIA TEXTIL,2025,6.006406,Moderado
5,51.466.860/0001-56,SAO MARTINHO S.A.,2025,5.778898,Moderado
6,12.528.708/0001-07,AERIS IND. E COM. DE EQUIP. PARA GER. DE ENG. ...,2024,0.472357,Critico
7,42.771.949/0001-35,ALLIANÇA SAÚDE E PARTICIPAÇÕES S.A.,2024,2.372012,Critico
8,60.537.263/0001-66,"ALLPARK EMPREENDIMENTOS, PARTICIPAÇÕES E SERVI...",2024,1.264868,Critico
9,61.079.117/0001-05,ALPARGATAS S.A.,2024,6.272027,Moderado
